# Motivating Example: TTA 이득/손해 구간을 정답 없이 구분할 수 있는가

런타임 > 런타임 유형 변경 > GPU (T4 이상, ViT-L은 A100 권장)

## 0. 코드 준비

레포가 비공개라 clone 대신 아래 셀들이 코드 파일을 Colab에 직접 만든다. 위에서부터 그대로 실행하면 된다.

In [ ]:
!mkdir -p /content/motivating
%cd /content/motivating
!pip -q install open_clip_torch gdown

In [ ]:
%%writefile corruptions.py
"""ImageNet-C 공식 수식(Hendrycks & Dietterich, 2019)을 torch로 옮긴 일부 손상.

입력은 [0,1] 범위의 float 텐서 (B,3,224,224). 같은 seed면 같은 손상이 나온다.
"""
import numpy as np
import torch
import torch.nn.functional as F
import torchvision

DOMAINS = ["clean", "gaussian_noise", "shot_noise", "defocus_blur", "contrast", "pixelate", "jpeg"]


def _gen(x, seed):
    g = torch.Generator(device=x.device)
    g.manual_seed(int(seed))
    return g


def gaussian_noise(x, s, seed):
    c = [0.08, 0.12, 0.18, 0.26, 0.38][s - 1]
    return (x + torch.randn(x.shape, generator=_gen(x, seed), device=x.device) * c).clamp(0, 1)


def shot_noise(x, s, seed):
    c = [60, 25, 12, 5, 3][s - 1]
    return (torch.poisson(x * c, generator=_gen(x, seed)) / c).clamp(0, 1)


def contrast(x, s, seed):
    c = [0.4, 0.3, 0.2, 0.1, 0.05][s - 1]
    m = x.mean(dim=(2, 3), keepdim=True)
    return ((x - m) * c + m).clamp(0, 1)


def _gauss1d(ksize, sigma):
    r = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2
    k = torch.exp(-(r ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def _disk(radius, alias_blur):
    if radius <= 8:
        L, ksize = np.arange(-8, 8 + 1), 3
    else:
        L, ksize = np.arange(-radius, radius + 1), 5
    X, Y = np.meshgrid(L, L)
    k = torch.tensor(((X ** 2 + Y ** 2) <= radius ** 2).astype(np.float32))
    k = k / k.sum()
    g = _gauss1d(ksize, alias_blur)
    g2 = (g[:, None] * g[None, :])[None, None]
    k = F.conv2d(F.pad(k[None, None], (ksize // 2,) * 4, mode="reflect"), g2)[0, 0]
    return k / k.sum()


def defocus_blur(x, s, seed):
    radius, alias = [(3, 0.1), (4, 0.5), (6, 0.5), (8, 0.5), (10, 0.5)][s - 1]
    k = _disk(radius, alias).to(x.device, x.dtype)
    p = k.shape[-1] // 2
    w = k[None, None].repeat(3, 1, 1, 1)
    return F.conv2d(F.pad(x, (p,) * 4, mode="reflect"), w, groups=3).clamp(0, 1)


def pixelate(x, s, seed):
    c = [0.6, 0.5, 0.4, 0.3, 0.25][s - 1]
    h = x.shape[-1]
    small = F.interpolate(x, size=int(h * c), mode="area")
    return F.interpolate(small, size=h, mode="nearest")


def jpeg(x, s, seed):
    q = [25, 18, 15, 10, 7][s - 1]
    u8 = (x * 255).round().clamp(0, 255).to(torch.uint8).cpu()
    out = [torchvision.io.decode_jpeg(torchvision.io.encode_jpeg(im, quality=q)) for im in u8]
    return torch.stack(out).to(x.device).float() / 255


_FNS = {
    "clean": lambda x, s, seed: x,
    "gaussian_noise": gaussian_noise,
    "shot_noise": shot_noise,
    "defocus_blur": defocus_blur,
    "contrast": contrast,
    "pixelate": pixelate,
    "jpeg": jpeg,
}


def corrupt(x, domain, severity, seed):
    return _FNS[domain](x, severity, seed)


In [ ]:
%%writefile methods.py
"""CLIP 이미지 인코더의 LayerNorm만 업데이트하는 온라인 TTA (Tent, ETA, SAR).

텍스트 분류기는 고정한다. 각 step은 업데이트 전 logits를 예측으로 돌려준다 (원 논문들과 같음).
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def entropy(logits):
    logp = logits.float().log_softmax(1)
    return -(logp.exp() * logp).sum(1)


class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, text_feat):
        super().__init__()
        self.visual = clip_model.visual
        self.register_buffer("text", text_feat)
        self.scale = clip_model.logit_scale.exp().item()

    def forward(self, x):
        f = F.normalize(self.visual(x).float(), dim=-1)
        return self.scale * f @ self.text.float().T


def ln_params(net, exclude_last_blocks=0):
    """visual 안의 LayerNorm affine만 학습. SAR처럼 위쪽 블록과 ln_post를 뺄 수 있다."""
    net.requires_grad_(False)
    n_blocks = len(net.visual.transformer.resblocks)
    params, names = [], []
    for name, m in net.visual.named_modules():
        if not isinstance(m, nn.LayerNorm):
            continue
        if exclude_last_blocks:
            if name == "ln_post":
                continue
            if name.startswith("transformer.resblocks."):
                if int(name.split(".")[2]) >= n_blocks - exclude_last_blocks:
                    continue
        for pn, p in m.named_parameters():
            p.requires_grad_(True)
            params.append(p)
            names.append(f"{name}.{pn}")
    return params, names


class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_cls, rho=0.05, **kw):
        super().__init__(params, dict(rho=rho, **kw))
        self.base = base_cls(self.param_groups, **kw)
        self.param_groups = self.base.param_groups

    @torch.no_grad()
    def first_step(self):
        grads = [p.grad.norm(2) for g in self.param_groups for p in g["params"] if p.grad is not None]
        norm = torch.norm(torch.stack(grads), 2) if grads else torch.tensor(0.0)
        for g in self.param_groups:
            scale = g["rho"] / (norm + 1e-12)
            for p in g["params"]:
                if p.grad is None:
                    continue
                self.state[p]["old_p"] = p.data.clone()
                p.add_(p.grad * scale)
        self.zero_grad()

    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if "old_p" in self.state[p]:
                    p.data = self.state[p].pop("old_p")
        self.base.step()
        self.zero_grad()


class Base:
    exclude_last_blocks = 0

    def __init__(self, net, lr, num_classes, amp=True):
        self.net, self.lr, self.C, self.amp = net, lr, num_classes, amp
        self.params, _ = ln_params(net, self.exclude_last_blocks)
        self.init_state = [p.detach().clone() for p in self.params]
        self.opt = self.make_opt()
        self.e_margin = 0.4 * math.log(num_classes)

    def make_opt(self):
        return torch.optim.SGD(self.params, lr=self.lr, momentum=0.9)

    def fwd(self, x):
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.amp):
            return self.net(x).float()

    def reset(self):
        with torch.no_grad():
            for p, p0 in zip(self.params, self.init_state):
                p.copy_(p0)
        self.opt = self.make_opt()
        self.on_reset()

    def on_reset(self):
        pass


class NoAdapt(Base):
    def __init__(self, net, lr, num_classes, amp=True):
        self.net, self.amp = net, amp
        net.requires_grad_(False)

    @torch.no_grad()
    def step(self, x):
        return self.fwd(x)

    def reset(self):
        pass


class Tent(Base):
    def step(self, x):
        logits = self.fwd(x)
        loss = entropy(logits).mean()
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return logits.detach()


class ETA(Base):
    """EATA에서 Fisher 정규화를 뺀 버전 (EATA 논문의 ETA)."""

    d_margin = 0.05

    def on_reset(self):
        self.ema_probs = None

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.ema_probs = None

    def step(self, x):
        logits = self.fwd(x)
        ent = entropy(logits)
        keep = ent < self.e_margin
        probs = logits.softmax(1)
        if self.ema_probs is not None:
            cos = F.cosine_similarity(self.ema_probs[None], probs, dim=1)
            keep = keep & (cos.abs() < self.d_margin)
        if keep.any():
            with torch.no_grad():
                m = probs[keep].mean(0).detach()
                self.ema_probs = m if self.ema_probs is None else 0.9 * self.ema_probs + 0.1 * m
            w = 1.0 / torch.exp(ent[keep].detach() - self.e_margin)
            loss = (ent[keep] * w).mean()
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
        return logits.detach()


class SAR(Base):
    """SAR (ICLR 2023): 신뢰 샘플만 + SAM + loss EMA가 0.2 아래면 모델 복구."""

    reset_constant = 0.2

    def __init__(self, net, lr, num_classes, amp=True):
        n = len(net.visual.transformer.resblocks)
        self.exclude_last_blocks = max(1, n // 4)  # ViT-B 12블록이면 원 논문처럼 위 3블록 제외
        super().__init__(net, lr, num_classes, amp)
        self.ema = None
        self.n_resets = 0

    def make_opt(self):
        return SAM(self.params, torch.optim.SGD, rho=0.05, lr=self.lr, momentum=0.9)

    def on_reset(self):
        self.ema = None

    def step(self, x):
        logits = self.fwd(x)
        ent = entropy(logits)
        keep = ent < self.e_margin
        if not keep.any():
            return logits.detach()
        ent[keep].mean().backward()
        self.opt.first_step()

        ent2 = entropy(self.fwd(x))[keep]
        keep2 = ent2 < self.e_margin
        if keep2.any():
            loss2 = ent2[keep2].mean()
            v = loss2.item()
            self.ema = v if self.ema is None else 0.9 * self.ema + 0.1 * v
            loss2.backward()
        self.opt.second_step()
        if self.ema is not None and self.ema < self.reset_constant:
            self.reset()
            self.n_resets += 1  # 붕괴 감지로 인한 복구만 센다
        return logits.detach()


METHODS = {"none": NoAdapt, "tent": Tent, "eta": ETA, "sar": SAR}


In [ ]:
%%writefile run.py
"""TTA 모델과 zero-shot 모델에 같은 스트림을 넣고 샘플마다 기록한다.

예: python run.py --model ViT-B-16 --method sar
결과: outputs/{model}_{method}_{stream}.npz (샘플 단위 기록, 구간 집계는 analyze.py)
"""
import argparse
import copy
import json
import os
import shutil
import tarfile
import time
import urllib.request

import numpy as np
import open_clip
import torch
import torch.nn.functional as F
from open_clip.zero_shot_metadata import IMAGENET_CLASSNAMES, SIMPLE_IMAGENET_TEMPLATES
from PIL import Image
from torchvision import transforms as T

from corruptions import DOMAINS, corrupt
from methods import METHODS, CLIPClassifier, entropy

V2_URL = "https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-matched-frequency.tar.gz"
V2_DRIVE_ID = "1HsymsaWYakVTbWaBzXSsGVk9Wf6j682X"
V2_DIR ="imagenetv2-matched-frequency-format-val"
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)


def load_v2(root, v2_tar=""):
    """V2 1만 장을 224x224 uint8로 한 번만 디코딩해서 캐시한다 (약 1.5GB).

    v2_tar: 마운트한 드라이브의 imagenetv2-matched-frequency.tar.gz 경로. 비우면 내려받는다.
    """
    cache_x, cache_y = os.path.join(root, "v2_224_x.npy"), os.path.join(root, "v2_224_y.npy")
    if os.path.exists(cache_x):
        return np.load(cache_x, mmap_mode="r"), np.load(cache_y)
    d = os.path.join(root, V2_DIR)
    if not os.path.isdir(d):
        os.makedirs(root, exist_ok=True)
        tgz = os.path.join(root, "v2.tar.gz")
        if v2_tar:
            print("드라이브에서 로컬 디스크로 복사 중:", v2_tar)
            shutil.copy(v2_tar, tgz)  # 드라이브에서 바로 풀면 느려서 먼저 복사
        else:
            print("ImageNet-V2 다운로드 중 (1.2GB)")
            try:  # 팀 드라이브 데이터셋 폴더에 있는 같은 파일
                import gdown
                gdown.download(id=V2_DRIVE_ID, output=tgz, quiet=False)
            except Exception as e:
                print("드라이브 다운로드 실패, HuggingFace에서 받음:", e)
                urllib.request.urlretrieve(V2_URL, tgz)
        with tarfile.open(tgz) as t:
            t.extractall(root)
        os.remove(tgz)
    tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224)])
    xs, ys = [], []
    for c in range(1000):
        cd = os.path.join(d, str(c))
        for f in sorted(os.listdir(cd)):
            im = tf(Image.open(os.path.join(cd, f)).convert("RGB"))
            xs.append(np.asarray(im).transpose(2, 0, 1))
            ys.append(c)
        if c % 200 == 0:
            print(f"  디코딩 {c}/1000")
    x, y = np.stack(xs), np.array(ys)
    np.save(cache_x, x)
    np.save(cache_y, y)
    return x, y


def dirichlet_order(idx, y, alpha, n_slots, rng):
    """RoTTA/ROID 방식의 클래스 쏠림 스트림. alpha가 작을수록 같은 클래스가 몰려 온다."""
    slots = [[] for _ in range(n_slots)]
    for c in np.unique(y):
        ids = rng.permutation(idx[y == c])
        cuts = (np.cumsum(rng.dirichlet([alpha] * n_slots)) * len(ids)).astype(int)[:-1]
        for s, part in enumerate(np.split(ids, cuts)):
            slots[s].extend(part.tolist())
    return np.array([i for s in slots for i in rng.permutation(s)], dtype=int)


@torch.no_grad()
def text_features(model, name, device, cache_dir):
    path = os.path.join(cache_dir, f"text_{name}.pt")
    if os.path.exists(path):
        return torch.load(path).to(device)
    tok = open_clip.get_tokenizer(name)
    feats = []
    for c in IMAGENET_CLASSNAMES:
        t = tok([tpl(c) for tpl in SIMPLE_IMAGENET_TEMPLATES]).to(device)
        f = F.normalize(model.encode_text(t).float(), dim=-1).mean(0)
        feats.append(F.normalize(f, dim=-1))
    feats = torch.stack(feats)
    torch.save(feats.cpu(), path)
    return feats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="ViT-B-16")
    ap.add_argument("--method", default="sar", choices=list(METHODS))
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--severity", type=int, default=5)
    ap.add_argument("--domains", default=",".join(DOMAINS))
    ap.add_argument("--n_per_domain", type=int, default=10000)
    ap.add_argument("--stream", default="dirichlet", choices=["iid", "dirichlet"])
    ap.add_argument("--alpha", type=float, default=0.01)
    ap.add_argument("--continual", action="store_true", help="도메인이 바뀌어도 모델을 초기화하지 않음")
    ap.add_argument("--grad_ckpt", action="store_true", help="메모리가 모자랄 때 (ViT-L)")
    ap.add_argument("--limit_batches", type=int, default=0, help="도메인당 batch 수 제한 (스모크 테스트)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--data_root", default="data")
    ap.add_argument("--v2_tar", default="", help="마운트한 드라이브의 V2 tar.gz 경로")
    ap.add_argument("--out", default="outputs")
    ap.add_argument("--tag", default="")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    dev = "cuda"
    os.makedirs(args.out, exist_ok=True)
    X, Y = load_v2(args.data_root, args.v2_tar)

    clip, _, _ = open_clip.create_model_and_transforms(args.model, pretrained="openai")
    clip = clip.to(dev).eval()
    text = text_features(clip, args.model, dev, args.out)
    if args.grad_ckpt:
        clip.set_grad_checkpointing(True)
    zs_net = CLIPClassifier(copy.deepcopy(clip), text).half().eval().requires_grad_(False)
    net = CLIPClassifier(clip, text).eval()
    del clip.transformer  # 텍스트 인코더는 더 안 씀
    method = METHODS[args.method](net, args.lr, 1000)

    mean, std = MEAN.to(dev), STD.to(dev)
    rng = np.random.default_rng(args.seed)
    rec = {k: [] for k in ["y", "dom", "pred_t", "pred_z", "ent_t", "ent_z", "maxp_t", "maxp_z",
                           "kl_tz", "pz_at_t", "pt_at_z"]}
    domains = args.domains.split(",")
    t0 = time.time()
    for di, dom in enumerate(domains):
        idx = rng.choice(len(Y), min(args.n_per_domain, len(Y)), replace=False)
        if args.stream == "dirichlet":
            idx = dirichlet_order(idx, Y[idx], args.alpha, max(1, len(idx) // args.bs), rng)
        if di > 0 and not args.continual:
            method.reset()
        nb = (len(idx) + args.bs - 1) // args.bs
        if args.limit_batches:
            nb = min(nb, args.limit_batches)
        correct_t = correct_z = n = 0
        for b in range(nb):
            sel = idx[b * args.bs:(b + 1) * args.bs]
            x = torch.from_numpy(np.ascontiguousarray(X[sel])).to(dev).float() / 255
            x = corrupt(x, dom, args.severity, seed=args.seed * 100003 + di * 1009 + b)
            x = (x - mean) / std
            y = torch.from_numpy(Y[sel]).to(dev)

            with torch.no_grad():
                lz = zs_net(x.half()).float()
            lt = method.step(x).float()

            with torch.no_grad():
                pt, pz = lt.softmax(1), lz.softmax(1)
                at, az = pt.argmax(1), pz.argmax(1)
                kl = (pt * (pt.clamp_min(1e-12).log() - pz.clamp_min(1e-12).log())).sum(1)
                vals = dict(y=y, dom=torch.full_like(y, di), pred_t=at, pred_z=az,
                            ent_t=entropy(lt), ent_z=entropy(lz), maxp_t=pt.max(1).values,
                            maxp_z=pz.max(1).values, kl_tz=kl,
                            pz_at_t=pz.gather(1, at[:, None])[:, 0], pt_at_z=pt.gather(1, az[:, None])[:, 0])
                for k, v in vals.items():
                    rec[k].append(v.cpu().numpy())
                correct_t += (at == y).sum().item()
                correct_z += (az == y).sum().item()
                n += len(y)
        print(f"[{dom:15s}] TTA {100 * correct_t / n:5.1f}  zero-shot {100 * correct_z / n:5.1f}  "
              f"차이 {100 * (correct_t - correct_z) / n:+5.1f}  ({time.time() - t0:.0f}s)", flush=True)

    name = f"{args.model}_{args.method}_{args.stream}{'_continual' if args.continual else ''}{args.tag}"
    out = {k: np.concatenate(v) for k, v in rec.items()}
    meta = dict(vars(args), domains=domains, n_resets=getattr(method, "n_resets", 0))
    np.savez_compressed(os.path.join(args.out, name + ".npz"), meta=json.dumps(meta), **out)
    print("저장:", os.path.join(args.out, name + ".npz"), "| SAR 복구 횟수:", meta["n_resets"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze.py
"""구간마다 정확도 차이(TTA - zero-shot)와 정답 없는 신호를 모아 분리력을 본다.

예: python analyze.py --seg 512
결과: outputs/segments.csv, outputs/separability.csv, outputs/scatter.png
"""
import argparse
import glob
import json
import math
import os

import numpy as np
import pandas as pd

LN_C = math.log(1000)

# 이름: (그룹, 샘플 단위 값을 만드는 함수). 모두 정답 없이 계산된다.
SIGNALS = {
    # TTA 모델만 보는 기존 신호
    "tta_entropy": ("existing", lambda r: r["ent_t"]),
    "tta_maxprob": ("existing", lambda r: r["maxp_t"]),
    "tta_reliable_frac": ("existing", lambda r: (r["ent_t"] < 0.4 * LN_C).astype(float)),
    # zero-shot과 비교하는 신호
    "disagree_rate": ("zs_compare", lambda r: (r["pred_t"] != r["pred_z"]).astype(float)),
    "kl_tta_zs": ("zs_compare", lambda r: r["kl_tz"]),
    "zs_prob_of_tta_pred": ("zs_compare", lambda r: r["pz_at_t"]),
    "conf_gap_on_disagree": ("zs_compare", None),  # 구간 단위로 따로 계산
    # 데이터 난이도 (zero-shot만 봄)
    "zs_entropy": ("data", lambda r: r["ent_z"]),
}


def auroc(score, label):
    """label=1(손해)을 score가 높을수록 잘 고르는 정도. 0.5면 무작위."""
    pos, neg = score[label == 1], score[label == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    ranks = pd.Series(np.concatenate([pos, neg])).rank().values
    return (ranks[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def segments(path, seg):
    z = np.load(path)
    meta = json.loads(str(z["meta"]))
    r = {k: z[k] for k in z.files if k != "meta"}
    run = os.path.basename(path)[:-4]
    rows = []
    for di, dom in enumerate(meta["domains"]):
        ids = np.where(r["dom"] == di)[0]
        for s in range(0, len(ids), seg):
            j = ids[s:s + seg]
            if len(j) < seg // 2:
                continue
            sub = {k: v[j] for k, v in r.items()}
            acc_t = (sub["pred_t"] == sub["y"]).mean()
            acc_z = (sub["pred_z"] == sub["y"]).mean()
            row = dict(run=run, model=meta["model"], method=meta["method"], domain=dom, seg=s // seg,
                       n=len(j), acc_tta=acc_t, acc_zs=acc_z, delta=acc_t - acc_z)
            for name, (_, fn) in SIGNALS.items():
                if fn is not None:
                    row[name] = float(np.mean(fn(sub)))
            d = sub["pred_t"] != sub["pred_z"]
            # 다르게 답한 샘플에서 zero-shot이 자기 답을 TTA 답보다 얼마나 더 믿는지
            row["conf_gap_on_disagree"] = float(np.mean(sub["maxp_z"][d] - sub["pz_at_t"][d])) if d.any() else 0.0
            rows.append(row)
    return rows


def separability(df):
    out = []
    harm = (df["delta"] < 0).astype(int).values
    for name, (group, _) in SIGNALS.items():
        a = auroc(df[name].values, harm)
        rho = df[[name, "delta"]].rank().corr().iloc[0, 1]
        out.append(dict(signal=name, group=group, auroc_harm=a,
                        separability=max(a, 1 - a) if not np.isnan(a) else np.nan, spearman_delta=rho))
    return pd.DataFrame(out)


def judge(sep):
    def best(g):
        s = sep[sep.group == g].separability
        return s.max() if len(s) else np.nan

    e, z = best("existing"), best("zs_compare")
    tag = lambda v: "나뉨" if v >= 0.8 else ("섞임" if v <= 0.65 else "애매")
    return f"기존 신호 최고 {e:.2f} ({tag(e)}) / zero-shot 비교 최고 {z:.2f} ({tag(z)})"


def plot(df, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    names = list(SIGNALS)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    runs = sorted(df.run.unique())
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(runs), 1)))
    for ax, name in zip(axes.flat, names):
        for c, run in zip(colors, runs):
            d = df[df.run == run]
            ax.scatter(d[name], 100 * d["delta"], s=12, alpha=0.7, color=c, label=run)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_title(f"[{SIGNALS[name][0]}] {name}", fontsize=10)
        ax.set_ylabel("acc(TTA) - acc(zero-shot) [%p]")
    axes.flat[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=130)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="outputs")
    ap.add_argument("--seg", type=int, default=512, help="구간 하나의 샘플 수")
    args = ap.parse_args()

    rows = []
    for p in sorted(glob.glob(os.path.join(args.out, "*.npz"))):
        rows += segments(p, args.seg)
    df = pd.DataFrame(rows)
    df = df[df.method != "none"]
    df.to_csv(os.path.join(args.out, "segments.csv"), index=False)

    pd.set_option("display.width", 200)
    print("\n== 실행별 구간 수와 손해 구간 비율 ==")
    print(df.groupby("run").agg(n_seg=("delta", "size"), harm_frac=("delta", lambda d: (d < 0).mean()),
                                mean_delta=("delta", "mean")).round(3))
    print("\n== 도메인별 평균 차이 (%p) ==")
    print((100 * df.pivot_table(index="run", columns="domain", values="delta")).round(1))

    seps = []
    for scope, d in [("pooled", df)] + [(r, g) for r, g in df.groupby("run")]:
        harm = (d.delta < 0).mean()
        if harm == 0 or harm == 1:
            print(f"\n[{scope}] 손해 구간 비율 {harm:.0%}: 이득과 손해가 섞여 있지 않아 분리력을 볼 수 없음")
            continue
        s = separability(d)
        s.insert(0, "scope", scope)
        seps.append(s)
        print(f"\n== [{scope}] 분리력 (손해 구간 비율 {harm:.0%}) ==")
        print(s.drop(columns="scope").round(3).to_string(index=False))
        print("판정:", judge(s))
    if seps:
        pd.concat(seps).to_csv(os.path.join(args.out, "separability.csv"), index=False)
    plot(df, os.path.join(args.out, "scatter.png"))
    print("\n그림:", os.path.join(args.out, "scatter.png"))


if __name__ == "__main__":
    main()


In [ ]:
# 공유 데이터셋 폴더를 먼저 '내 드라이브'에 바로가기로 추가해야 여기서 보인다
# (드라이브에서 폴더 우클릭 > 정리 > 바로가기 추가 > 내 드라이브)
from google.colab import drive
drive.mount('/content/drive')
V2_TAR = '/content/drive/MyDrive/<바로가기 폴더 이름>/imagenetv2-matched-frequency.tar.gz'
import os; print('V2 파일 있음' if os.path.exists(V2_TAR) else '경로를 확인하세요: ' + V2_TAR)

## 1. 실행

- 첫 실행에서 마운트한 드라이브의 V2 tar를 로컬로 복사해 풀고 224px 캐시를 만든다 (몇 분)
- 결과는 `outputs/`에 실행마다 npz 하나씩
- 시간이 부족하면 `--n_per_domain 5000`

In [ ]:
!python run.py --model ViT-B-16 --method sar --v2_tar "$V2_TAR"  # 첫 실행에서 data/ 캐시 생성, 이후 실행은 캐시 사용

In [ ]:
!python run.py --model ViT-B-16 --method tent

In [ ]:
!python run.py --model ViT-L-14 --method sar --grad_ckpt

In [ ]:
!python run.py --model ViT-L-14 --method eta --grad_ckpt

## 2. 분석

In [ ]:
!python analyze.py --seg 512
from IPython.display import Image
Image('outputs/scatter.png')

## 3. 결과 보관 (선택)

Colab 런타임이 끊기면 outputs가 사라지니 내 드라이브로 복사해 둔다.

In [ ]:
!mkdir -p /content/drive/MyDrive/motivating_outputs && cp outputs/*.npz outputs/*.csv outputs/*.png /content/drive/MyDrive/motivating_outputs/